<a href="https://colab.research.google.com/github/vignesh23450/3D_printer/blob/main/Part1/Basic_Agent_Part_1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: August 2026
(Tested on Google Colab)

# ⚠️ Important: If running in Colab , Switch to Python 3.12

Before running any other cells in this notebook, please switch your Google Colab runtime to **Python 3.12**.

From the top right arrow (next to Connect/RAM), select:

**Change runtime type → Runtime Version → 2026.07**

The next cell should output python version as 3.12


In [1]:
!python --version

Python 3.12.13


# IMPORTANT

1. Run the installation cell first
2. Add required API keys in Colab Secrets
3. Run notebook cells sequentially

# If running in Google Colab Please ensure you update below keys in "secrets" on the left and give access to this notebook

1.   OPENAI_API_KEY
2.   TAVILY_API_KEY

> 💡 **Note:** While running package installation command below,Google Colab may display some dependency messages during installation. These relate to packages we are not using in this notebook, so you can safely continue when the installation completes.

In [ ]:
!pip install -q \
    "langchain==0.3.14" \
    "langchain-openai==0.2.14" \
    "langchain-community==0.3.14" \
    "langchain-core==0.3.63" \
    "openai==1.59.6" \
    "python-dotenv==1.0.1" \
    "requests==2.32.4" \
    "beautifulsoup4>=4.13.0" \
    "wikipedia==1.4.0" \
    "tavily-python==0.5.0"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.9/326.9 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.9 requires langchain-core<2,>=1.4.7, but you have langchain-c

In [ ]:
#below lines are needed for restarting kernel in Colab -
# If you are in local environment you can use "restart" button on top of the notebook
import os
os.kill(os.getpid(), 9)

In [ ]:
#Langchain
from langchain.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, AgentType
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
#If executing from local machine, run below 2 lines to load keys (.env should be present in same directory with keys in it)
# from dotenv import load_dotenv
# load_dotenv()


#If executing from Colab, run below lines to load keys (keys should be added in colab secrets and access given to this notebook)
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#prompt templates
prompt_template = PromptTemplate(
    input_variables=["name"],
    template="Hello, {name}! How can I help you today?"
)

formatted_prompt = prompt_template.format(name="David")
print(formatted_prompt)

Hello, David! How can I help you today?


In [ ]:
chat_model = ChatOpenAI(model="gpt-4o-mini")

response = chat_model.invoke("What is Capital of USA?")
print(response.content)

The capital of the United States is Washington, D.C.


In [ ]:
#LLM Chains
llm = ChatOpenAI(model="gpt-4o-mini")

learn_template = """
I want you to act as a consultant for a AI training
Return a list of topics and why it is important to learn in given area of AI
The description should be relevant to recent advancement in AI
What are some good topics to learn in {AI_topic}
"""

learn_prompt = PromptTemplate(
    input_variables=["AI_topic"],
    template=learn_template,
)

description = "Deep learning"

chain = LLMChain(llm=llm, prompt=learn_prompt)

result = chain.invoke({"AI_topic": description})
print(result["text"])

Absolutely! Deep learning is a rapidly evolving field within artificial intelligence (AI), and staying up-to-date with recent advancements is crucial for anyone looking to work in this area. Here are some key topics to consider learning, along with their relevance and importance:

### 1. **Neural Network Architectures**
   - **Importance**: Understanding different architectures (e.g., CNNs, RNNs, Transformers) is essential for solving various types of problems. For instance, CNNs are crucial for image processing, while Transformers have revolutionized natural language processing tasks.
   - **Recent Advancements**: The development of architectures like Vision Transformers (ViT) and the resurgence of recurrent networks show the diversity and specialization in neural network designs.

### 2. **Transfer Learning**
   - **Importance**: Transfer learning allows practitioners to leverage pre-trained models on large datasets, significantly reducing the time and data required to train models f

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool

# Define a simple custom tool
def my_tool_function(query: str) -> str:
    return f"Tool response: {query}"

# Create tool from function
my_tool = Tool.from_function(
    func=my_tool_function,
    name="simple_tool",
    description="A simple tool"
)

# Tavily Search Tool
tavily_search = TavilySearchResults(max_results=2)

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Tools list
tools = [tavily_search, my_tool]

# Create agent
agent = initialize_agent(
    tools,
    llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
response = agent.run("What's the weather like today in London?")

print(response)



> Entering new AgentExecutor chain...
I need to find the current weather information for London. Since this is a current event, I'll use the search engine to get accurate and up-to-date results.  
Action: tavily_search_results_json  
Action Input: "current weather in London"  
Observation: [{'url': 'https://www.accuweather.com/en/gb/london/ec4a-2/august-weather/328328', 'content': '# London, London\n\nLondon\n\nLondon\n\n## Around the Globe\n\nAround the Globe\n\n### Radar & Maps\n\n### News & Features\n\n### Astronomy\n\n### Business\n\n### Climate\n\n### Health\n\n### Recreation\n\n### Sports\n\n### Travel\n\n### Warnings\n\n### Data Suite\n\n### Forensics\n\n### Advertising\n\n### Superior Accuracy™\n\n### Video\n\n### Severe Weather\n\n### Hurricane Tracker\n\n## Monthly\n\n## August\n\n## 2026\n\n## 10-Day\n\nPartly sunny and beautiful\nBeautiful with periods of sun\nMostly cloudy and comfortable\nMostly cloudy, a shower; warm\nA little morning rain\nA couple of showers\nSome su

In [ ]:
prompt_template = "Summarize the following content: {content}"
llm = ChatOpenAI(model="gpt-4o-mini")

llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(prompt_template)
)

summarize_tool = Tool.from_function(
    func=llm_chain.run,
    name="Summarizer",
    description="Summarizes a web page"
)

In [ ]:
tools = [tavily_search, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,handle_parsing_errors=True
)

In [ ]:
response = agent.invoke({"input": "Who invented the World Wide Web and what impact did it have?"})
print(response["output"])



> Entering new AgentExecutor chain...
I need to gather information about the inventor of the World Wide Web and its impact. I will start by searching for recent articles or summaries that provide this information. 

Action: tavily_search_results_json
Action Input: "Who invented the World Wide Web and what impact did it have?"

Observation: [{'url': 'https://en.wikipedia.org/wiki/Tim_Berners-Lee', 'content': 'Sir Timothy John Berners-Lee (born 8 June 1955), also known as TimBL, is an English computer scientist best known as the inventor of the World Wide Web, HTML, the URL system, and HTTP. He is a professorial research fellow at the University of Oxford and a professor emeritus at the Massachusetts Institute of Technology (MIT). [...] | Sir Tim Berners Lee arriving at the Guildhall to receive the Honorary Freedom of the City of London.jpg) Berners-Lee in 2025 |\n| Born | Timothy John Berners-Lee   (1955-06-08) 8 June 1955 (age 71)  London, England |\n| Other names |  TimBL  TBL |\n| 